# 02 — Engine Basics

In notebook 00 you wrote a `chat()` function that posts JSON to Ollama and parses the response. It works. It's also the exact wrong abstraction if you're building something real.

The problem: every provider has a different API. Anthropic expects `max_tokens`, OpenAI wants `max_completion_tokens`, Ollama ignores both. Error shapes differ. Response fields differ. Your `chat()` function is hardcoded to one provider — Ollama — and adding a second means duplicating the logic with conditionals everywhere.

`llm_engines` solves this once. One interface, swappable backends, normalized responses. This notebook shows you how to use it and why it's structured the way it is.

## The engine factory

Instead of calling HTTP directly, you ask the factory for an engine instance. The engine knows how to talk to its backend. Your code talks to the engine.

In [2]:
from llm_engines import get_engine
from llm_engines.contracts import GenerationRequest

# Get an Ollama engine
engine = get_engine("ollama", "qwen3.5:9b")  # adjust to your pulled model

# Make a request
request = GenerationRequest(
    messages=[
        {"role": "system", "content": "You are terse. One sentence."},
        {"role": "user",   "content": "What is a context window?"},
    ],
    temperature=0.7,
)

response = engine.generate(request)
print(response.message.content)

A context window is the maximum amount of text, measured in tokens, that a language model can process and remember at once to generate a response.


That's the core pattern. `get_engine(backend, model)` returns an object with a `.generate()` method. The method takes a `GenerationRequest`, returns a `GenerationResponse`. The contracts are in `llm_engines.contracts` — read them. They're small.

## What's in the response

The response object has normalized fields that work the same regardless of backend:

In [3]:
print(f"Content: {response.message.content}")
print(f"Finish reason: {response.finish_reason}")  # 'stop', 'length', 'error'
print(f"Model: {response.model_name}")
print(f"Tokens (input): {response.usage.input_tokens}")
print(f"Tokens (output): {response.usage.output_tokens}")
print(f"Tokens (total): {response.usage.total_tokens}")
print(f"Latency: {response.usage.latency_ms}ms")

Content: A context window is the maximum amount of text, measured in tokens, that a language model can process and remember at once to generate a response.
Finish reason: stop
Model: qwen3.5:9b
Tokens (input): 30
Tokens (output): 30
Tokens (total): 60
Latency: 462.977ms


These fields exist on every response, regardless of whether the backend is Ollama, Anthropic, OpenAI, or vLLM. Your downstream code — the part that does something with the response — doesn't care which model produced it. That's the point.

## Swapping backends

Change one line. Everything else stays the same.
**Note:** The Anthropic engine requires `ANTHROPIC_API_KEY` in your environment. 
If the cell below says "ANTHROPIC_API_KEY not set," the variable isn't reaching 
the kernel. Fix: restart Jupyter after setting the variable in your shell, or 
set it directly in the notebook with `os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'`.

In [4]:
# Swap to Anthropic (requires ANTHROPIC_API_KEY in environment)
import os

if os.getenv("ANTHROPIC_API_KEY"):
    anthropic_engine = get_engine("anthropic", "claude-sonnet-4-20250514")
    
    # Same request, different backend
    response = anthropic_engine.generate(request)
    print(f"Anthropic response: {response.message.content}")
    print(f"Finish reason: {response.finish_reason}")
    print(f"Tokens: {response.usage.total_tokens}")
else:
    print("ANTHROPIC_API_KEY not set — skipping cloud comparison.")

Anthropic response: A context window is the maximum amount of text (measured in tokens) that an AI model can process and remember at one time during a conversation or task.
Finish reason: stop
Tokens: 55


The `GenerationRequest` object is identical. The response shape is identical. The only difference is which backend processed it. This is what makes A/B testing tractable — you can route the same prompt to two models and compare outputs without rewriting the caller.

## Structured output

When the next thing in your pipeline needs to parse the response, ask for JSON and validate it. `llm_engines` handles the common failure modes automatically (JSON wrapped in markdown fences, trailing commas, etc.).

In [5]:
import json

# Request JSON output
request = GenerationRequest(
    messages=[
        {"role": "system", "content":
         "Extract invoice data as JSON. Schema: {vendor: str, amount: float, currency: str}. "
         "Respond ONLY with the JSON object."},
        {"role": "user", "content":
         "Invoice from Widgets Inc, total USD 450.00."},
    ],
    temperature=0.0,
)

response = engine.generate(request)

# Parse the JSON response
try:
    data = json.loads(response.message.content)
    print("Parsed:", data)
except json.JSONDecodeError as e:
    print(f"Parse failed: {e}")
    print(f"Raw output: {response.message.content}")

Parsed: {'vendor': 'Widgets Inc', 'amount': 450.0, 'currency': 'USD'}


For production use, you'd wrap this in retry logic — if the parse fails, append `"The JSON was malformed. Try again."` to the conversation and regenerate. Notebook 06 shows this pattern in detail when building RAG pipelines that need structured extraction.

## Token accounting

Every response includes token counts. This lets you:
- Budget API costs (cloud providers charge per token)
- Detect runaway prompts before they hit the model
- Measure how much context you're actually using

Example: measure the cost of a large prompt.

In [17]:
# Simulate a large prompt (real use case: building context from retrieved docs)
large_context = "Background info: " + ("x" * 50000)  # ~12.5K tokens

request = GenerationRequest(
    messages=[
        {"role": "system", "content": large_context},
        {"role": "user",   "content": "Summarize the background."},
    ],
    temperature=0.7,
)

response = engine.generate(request)
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Latency: {response.usage.latency_ms}ms")

# In real code, you'd check token count before calling generate() and truncate if needed.
# Notebooks 04-06 show context assembly strategies that respect token budgets.

Input tokens: 7362
Output tokens: 64
Total tokens: 7426
Latency: 1128.297ms


## Error handling

The engine raises exceptions on failures. Catch them at the caller level, not inside the engine. 
All exception types are in `llm_engines.contracts`.

In [19]:
from llm_engines.contracts import GenerationError, RateLimitError, LLMEngineError

try:
    response = engine.generate(request)
except RateLimitError as e:
    print(f"Rate limit hit: {e}. Back off and retry.")
except GenerationError as e:
    print(f"Model error: {e}. Check the request.")
except LLMEngineError as e:
    print(f"LLM engine error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

The exception hierarchy in `llm_engines.contracts` includes: `RateLimitError`, `GenerationError`, `ModelNotFoundError`, `ContextLengthExceededError`, `BackendUnavailableError`, and the base `LLMEngineError`. Catch the specific types your code can recover from; let the rest bubble up.

## When to skip the engine layer

The engine is the right abstraction most of the time. It's the wrong abstraction when:

- **You're debugging a provider-specific issue.** Drop to raw HTTP, read the actual error, fix it.
- **You're testing a new provider not yet in `llm_engines`.** Write raw HTTP first, port to engine later.
- **You need a provider-specific feature the engine doesn't expose yet.** Either extend the engine or use raw HTTP as a temporary workaround.

The engine is a *convenience layer*, not a requirement. If it gets in the way, go around it. The worst codebases are the ones that force abstraction everywhere even when direct calls are clearer.

## Artifact: `chat()` rewritten

Here's your notebook 00 `chat()` function, rewritten with `llm_engines`. Same behavior, swappable backends, real error handling.

In [21]:
from llm_engines import get_engine
from llm_engines.contracts import GenerationRequest, LLMEngineError

def chat(
    messages: list[dict],
    *,
    backend: str = "ollama",
    model: str = "qwen3.5:9b",
    temperature: float = 0.7,
) -> str:
    """Send a chat request to any backend. Returns the text response."""
    engine = get_engine(backend, model)
    request = GenerationRequest(messages=messages, temperature=temperature)
    
    try:
        response = engine.generate(request)
        return response.message.content
    except EngineLLMError as e:
        raise RuntimeError(f"LLM call failed: {e}") from e

# Test it
result = chat(
    [{"role": "user", "content": "Explain what an API abstraction layer does."}],
    temperature=0.7,
)
print(result)

An **API Abstraction Layer** (often called an API facade, wrapper, or adapter) is a software component that sits between your application's internal code and external third-party services (like payment gateways, cloud providers, or databases). Its primary goal is to **hide the complexity and volatility of external systems** from your core application.

Here is a breakdown of what it does and why it is essential:

### 1. Hides Complexity and Variance
Different external APIs have different requirements. For example:
*   **Stripe** might require a POST request with a specific JSON structure to charge a card.
*   **Twilio** might use a different endpoint and authentication method for sending SMS.
*   **AWS S3** uses a completely different protocol (REST/S3) than a standard HTTP REST API.

Without an abstraction layer, your application would need to write specific code for each of these distinct behaviors. The abstraction layer provides a **single, unified interface**. Your code simply call

Fourteen lines. Swap `backend="anthropic"` and it routes to Claude instead of Ollama. Token accounting, error types, normalized responses — all work the same regardless of which backend you're hitting.

## What's next

You now have:
- A working `chat()` function that abstracts providers
- Knowledge of the `GenerationRequest` / `GenerationResponse` contracts
- Token accounting you can use to budget context assembly

Everything from here on uses `llm_engines` under the hood. Notebook 03 shows you how to **inspect what actually happened** — what reached the model, what came back, why two runs differ. That's where the trace events this engine emits become useful.

## Exercises

1. **Swap backends mid-conversation.** Send the first three turns to Ollama, route turn 4 to Anthropic with the full history. Compare the responses. What differences show up?

2. **Token budget enforcement.** Write a wrapper around `chat()` that refuses requests over 10K tokens. Hint: count tokens in the messages before calling the engine.

3. **Structured extraction with retry.** Build a JSON extractor that retries up to 3 times if the parse fails, appending `"The JSON was malformed. Try again."` to the conversation. Measure how often the retry succeeds.

4. **Compare costs.** Run the same 100-message workload through Ollama (local) and Anthropic (cloud). Measure tokens and calculate the API cost difference at current pricing.

---
**Next:** [03 — Inspecting Model Behavior](03_inspecting_model_behavior.ipynb) introduces `llm_inspector`, which turns every `generate()` call into an inspectable trace.